In [0]:
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"])
write_mode = dbutils.widgets.get("write_mode")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
jdbc_url = "jdbc:sqlserver://quantcloud.database.windows.net:1433;database=quant-cloud-db"
user = dbutils.secrets.get(scope = "my-qct-sc", key = "sql-login-user")
password = dbutils.secrets.get(scope = "my-qct-sc", key = "sql-password")

connection_props = {
    "user": user,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
# cust_df = (
#     spark.read.format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", "SalesLT.Customer")
#     .options(**connection_props)
#     .load()
# )

# cust_df.display()

In [0]:
info_query = """
    SELECT 
    TABLE_SCHEMA, 
    TABLE_NAME 
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'SalesLT'
"""

info_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("query", info_query)
    .options(**connection_props)
    .load()
)

info_df.display()

In [0]:
for row in info_df.collect():
    print(f"Loading -- {row.TABLE_SCHEMA}.{row.TABLE_NAME} in {write_mode} mode")
    source_table = f"{row.TABLE_SCHEMA}.{row.TABLE_NAME}"
    target_table = f"{(row.TABLE_NAME).lower()}_{datetime.now().strftime("%Y%m%d")}"

    try:
        df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", source_table)
            .options(**connection_props)
            .load()
        )

        (
            df.write.format("parquet")
            .mode(write_mode)
            .save(f"abfss://adb-container@quantcloudadls.dfs.core.windows.net/sql_data/{target_table}")
        )
        print(f"Loading Success -- {target_table}")
    except Exception as e:
        print(f"Error: {e}")
    

In [0]:
# dbutils.fs.ls("abfss://adb-container@quantcloudadls.dfs.core.windows.net/")